In [ ]:
import idx2numpy
import numpy as np

with open("archive/t10k-images.idx3-ubyte", "rb") as f:
    test_images = idx2numpy.convert_from_file(f)

with open("archive/t10k-labels.idx1-ubyte", "rb") as f:
    test_lables = idx2numpy.convert_from_file(f)


# 1. Load raw files
train_images = idx2numpy.convert_from_file("archive/train-images.idx3-ubyte")
train_labels = idx2numpy.convert_from_file("archive/train-labels.idx1-ubyte")
train_images = train_images[:501]
train_labels = train_labels[:501]

# 2. Reshape to NCHW (N, 1, 28, 28) and scale to float32 [0.0, 1.0]
train_images = train_images[:, np.newaxis, :, :].astype(np.float32) / 255.0

# Helper function to generate mini-batches
def get_batches(X, y, batch_size=32, shuffle=True):
    num_samples = X.shape[0]
    indices = np.arange(num_samples)
    if shuffle:
        np.random.shuffle(indices)
    
    for start_idx in range(0, num_samples, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]
        yield X[batch_idx], y[batch_idx]

In [ ]:
import sys
import numpy as np
from pathlib import Path

curr_dir = Path.cwd()
prj_root = curr_dir.parent.parent
sys.path.append(str(prj_root))

from core.model import Sequential
from core.Nexus import Nexus
from extensions.Layers import Conv2D, Linear, MaxPool2D, Flatten
from extensions.Activations import ReLU, Softmax
from extensions.Optimizers import SGD
from extensions.Loss import CategoricalCrossEntropyLoss


In [ ]:

model = Sequential(
    # (N, 1,28,28) -> (N,4,28,28)
    Conv2D(3, 1,4,1,1),
    # (N,4,28,28)
    ReLU(),
    # (N,4,14,14)
    MaxPool2D(2,2),
    # (N, 4*14*14)
    Flatten(),
    # (N, 10)
    Linear(4*14*14,10),
)

optim = SGD(model.parameters(), lr=0.01)

In [ ]:
epochs = 500
batch_size = 64
losses = []

num_classes = 10

for epoch in range(epochs):
    avg_loss=[]
    
    for bx_np, by_np in get_batches(train_images, train_labels, batch_size=batch_size):
        optim.zero_grad()

        # Wrap numpy batch into Nexus Node
        bx = Nexus(bx_np)
        by = Nexus(np.eye(num_classes)[by_np])
        # Forward Pass
        y_logits = model.forward(bx)

        # Compute Loss (expects raw logits and integer target indices)
        loss = CategoricalCrossEntropyLoss(by, y_logits)

        # Backward Pass & Update
        loss.backward()
        optim.step()

        avg_loss.append(loss.value.item())
        
    losses.append(np.mean(avg_loss))
    if epoch%50==0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {np.mean(avg_loss):.8f}")

model.export_weights("cnn_model")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(epochs), losses)
plt.title("Categorical Cross Entropy Loss Optimization Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()